# HARs Burden Test - EUR

Corre únicamente burden tests con RVTests sobre ancestría EUR, partiendo de los VCFs  ya generados por `regionExtractor.py` 


A-No se corre ANNOVAR. Las HARs son no-codificantes, así que las clases que dependen de anotación funcional (`coding_variants`, `lof`, `coding_nonsynonymous`, `potentially_functional`) y la clase `potencially_pathogenic` (basada en CLNSIG/REVEL/CADD) no aplican o están sesgadas hacia variantes codificantes.
B-MAF filtering nativo en RVTests** vía `--freqUpper`, usando MAF interna de la cohorte (no gnomAD). Esto evita la necesidad de ANNOVAR.
C-Tres clases de burden: `all_variants`, `maf01` (MAF<0.01), `maf03` (MAF<0.03).
D-Custom HAR-refFlat: se construye un archivo `refFlat`-style donde cada HAR figura como un "gen" con sus coordenadas, para que `rvtest --gene HAR --geneFile har_refFlat` agrupe correctamente las variantes.
E-Datasets soportados: NBA y WGS (controlable con la lista `datasets`).

**Inputs esperados** (ya producidos por `regionExtractor.py` y por la celda de covariate del notebook original):
- VCFs por-HAR: `/home/jupyter/workspace/ws_files/Working_{set}/EUR/InputFiles/Indiv_HARS/{HAR}.vcf.gz`
- Covariate file: `/home/jupyter/workspace/ws_files/Working_{set}/EUR/InputFiles/EUR_covariate_file.txt`
- Lista de HARs: `/home/jupyter/workspace/ws_files/Original_Files/HAR_list_phase_1.tsv`

**Outputs**:
- Por-HAR: `/home/jupyter/workspace/ws_files/Novalis/Burden/{set}/EUR/{HAR}_{class}.burden.{Skat,SkatO}.assoc`
- Compilados: `/home/jupyter/workspace/ws_files/Novalis/Results/{set}_EUR_BURDEN.{SKAT,SKAT-O}.tsv`

## 1. Setup, imports y configuración

In [1]:
import os
import csv
import pathlib
import subprocess
from datetime import datetime
from multiprocessing import Manager
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from pathlib import Path

In [2]:
# ---------- Configuración ----------
release = 11

# >>> EUR ONLY <<<
ANCESTRY = "EUR"

# Dataset(s) a correr. Quitá el que no quieras.
datasets = ["NBA", "WGS"]

# Clases de burden (sin potencially_pathogenic)
BURDEN_CLASSES = {
    "all_variants": None,    # sin filtro de MAF
    "maf01":        0.01,    # --freqUpper 0.01
    "maf03":        0.03,    # --freqUpper 0.03
}

# Paralelismo
MAX_WORKERS = 16

# ---------- Paths ----------
DIR_TOOL  = "/home/jupyter/tools"
DIR_WSPS  = "/home/jupyter/workspace/ws_files"
DIR_ORIG  = f"{DIR_WSPS}/Original_Files"
DIR_NOVA  = f"{DIR_WSPS}/Novalis"          # raíz de OUTPUTS de Novalis
DIR_BURDEN = f"{DIR_NOVA}/Burden"
DIR_RESU   = f"{DIR_NOVA}/Results"

RVTEST  = f"{DIR_TOOL}/rvtests/executable/rvtest"
PLINK2  = f"{DIR_TOOL}/plink2"

# refFlat custom donde cada HAR es un "gen"
HAR_REFFLAT = f"{DIR_NOVA}/HAR_refFlat_hg38.txt"

# Crear directorios base
for d in [DIR_NOVA, DIR_BURDEN, DIR_RESU]:
    os.makedirs(d, exist_ok=True)
for ds in datasets:
    os.makedirs(f"{DIR_BURDEN}/{ds}/{ANCESTRY}", exist_ok=True)

print(f"Ancestry: {ANCESTRY}")
print(f"Datasets: {datasets}")
print(f"Burden classes: {list(BURDEN_CLASSES.keys())}")
print(f"Output root: {DIR_NOVA}")

Ancestry: EUR
Datasets: ['NBA', 'WGS']
Burden classes: ['all_variants', 'maf01', 'maf03']
Output root: /home/jupyter/workspace/ws_files/Novalis


## 2. Cargar lista de HARs

In [3]:
with open(f'{DIR_ORIG}/HAR_list_phase_1.tsv', 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    HARS_DICT = {
        row[3]: {
            'name':  row[3],
            'chrom': row[0].replace('chr', ''),
            'start': int(row[1]),
            'end':   int(row[2]),
        }
        for row in reader
    }
print(f"HARs cargados: {len(HARS_DICT)}")
print("Ejemplo:", next(iter(HARS_DICT.items())))

HARs cargados: 9476
Ejemplo: ('ZOOHAR.1', {'name': 'ZOOHAR.1', 'chrom': '2', 'start': 235865386, 'end': 235865436})


## 3. Construir refFlat custom para HARs

RVTests usa `--gene NAME --geneFile refFlat` para agrupar variantes por gene. Como las HARs no aparecen en el refFlat estándar, generamos uno propio donde cada HAR es un "gen" cuyas coordenadas cubren toda la región.

**Formato refFlat** (11 columnas, tab-separated):
`geneName  name  chrom  strand  txStart  txEnd  cdsStart  cdsEnd  exonCount  exonStarts  exonEnds`

In [4]:
with open(HAR_REFFLAT, 'w') as f:
    for HAR, info in HARS_DICT.items():
        chrom = info['chrom'] if info['chrom'].startswith('chr') else f"chr{info['chrom']}"
        start = info['start']
        end   = info['end']
        # geneName  name  chrom  strand  txStart  txEnd  cdsStart  cdsEnd  exonCount  exonStarts  exonEnds
        f.write('\t'.join([
            HAR, HAR, chrom, '+',
            str(start), str(end),
            str(start), str(end),
            '1',
            f'{start},',
            f'{end},'
        ]) + '\n')

print(f"HAR refFlat escrito en: {HAR_REFFLAT}")
!head -3 {HAR_REFFLAT}
!wc -l {HAR_REFFLAT}

HAR refFlat escrito en: /home/jupyter/workspace/ws_files/Novalis/HAR_refFlat_hg38.txt
ZOOHAR.1	ZOOHAR.1	chr2	+	235865386	235865436	235865386	235865436	1	235865386,	235865436,
ZOOHAR.2	ZOOHAR.2	chr13	+	72540830	72541395	72540830	72541395	1	72540830,	72541395,
ZOOHAR.3	ZOOHAR.3	chr5	+	77595138	77595554	77595138	77595554	1	77595138,	77595554,
9476 /home/jupyter/workspace/ws_files/Novalis/HAR_refFlat_hg38.txt


## 4. Verificar que existen los inputs (sanity check)

Verifica que para cada HAR exista el VCF + tbi de EUR generado por `regionExtractor.py`, y que exista el covariate file. Imprime los faltantes.


=== NBA ===
  covariate_ok: True
  covariate_path: /home/jupyter/workspace/ws_files/Working_NBA/EUR/InputFiles/EUR_covariate_file.txt
  n_vcfs_present: 8171
  n_vcfs_missing: 1305
  missing_examples: ['ZOOHAR.1', 'ZOOHAR.4', 'ZOOHAR.10', 'ZOOHAR.16', 'ZOOHAR.24']

=== WGS ===
  covariate_ok: True
  covariate_path: /home/jupyter/workspace/ws_files/Working_WGS/EUR/InputFiles/EUR_covariate_file.txt
  n_vcfs_present: 8895
  n_vcfs_missing: 581
  missing_examples: ['ZOOHAR.16', 'ZOOHAR.21', 'ZOOHAR.24', 'ZOOHAR.52', 'ZOOHAR.59']


In [ ]:
def vcf_path(set_name, HAR):
    return f"{DIR_WSPS}/Working_{set_name}/{ANCESTRY}/InputFiles/Indiv_HARS/{HAR}.vcf.gz"

def covar_path(set_name):
    return f"{DIR_WSPS}/Working_{set_name}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"

summary = {}
for ds in datasets:
    cov = covar_path(ds)
    cov_ok = os.path.isfile(cov)
    missing_vcfs = []
    for HAR in HARS_DICT:
        v = vcf_path(ds, HAR)
        if not (os.path.isfile(v) and os.path.isfile(v + '.tbi')):
            missing_vcfs.append(HAR)
    summary[ds] = {
        'covariate_ok': cov_ok,
        'covariate_path': cov,
        'n_vcfs_present': len(HARS_DICT) - len(missing_vcfs),
        'n_vcfs_missing': len(missing_vcfs),
        'missing_examples': missing_vcfs[:5],
    }

for ds, s in summary.items():
    print(f"\n=== {ds} ===")
    for k, v in s.items():
        print(f"  {k}: {v}")

**Nota**: si en `n_vcfs_missing` aparecen HARs, son los que `regionExtractor.py` marcó como SKIP (sin variantes que pasen filtros). Están listados en `Working_{set}/{ANCESTRY}/failed_HARs_{set}_{ANCESTRY}.tsv`. Esos HARs simplemente no van a tener burden test - es esperado.

## 5. Función de burden test (sin ANNOVAR)

Una sola función, cualquier MAF class la define `freq_upper`:
- `None` → sin filtro (clase `all_variants`)
- `0.01` → `--freqUpper 0.01` (clase `maf01`)
- `0.03` → `--freqUpper 0.03` (clase `maf03`)

In [ ]:
def burden_rvtest(set_name, HAR, burden_class, freq_upper, log_lock=None):
    """Corre RVTests SKAT/SKAT-O sobre el VCF de un HAR para EUR."""
    in_vcf  = vcf_path(set_name, HAR)
    covar   = covar_path(set_name)

    # Si no existe el VCF (HAR fue SKIP en regionExtractor), saltar
    if not os.path.isfile(in_vcf):
        return f"[SKIP] {set_name} {HAR} {burden_class} - no input VCF"

    out_dir = f"{DIR_BURDEN}/{set_name}/{ANCESTRY}"
    os.makedirs(out_dir, exist_ok=True)
    out_prefix = f"{out_dir}/{HAR}_{burden_class}.burden"

    cmd = [
        RVTEST,
        "--noweb",
        "--hide-covar",
        "--inVcf",     in_vcf,
        "--out",       out_prefix,
        "--kernel",    "skat,skato",
        "--pheno",     covar,
        "--pheno-name","PHENO",
        "--covar",     covar,
        "--covar-name","SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
        "--gene",      HAR,
        "--geneFile",  HAR_REFFLAT,
    ]
    if freq_upper is not None:
        cmd += ["--freqUpper", str(freq_upper)]

    ts = datetime.now().strftime('%H:%M:%S')
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=False)
        if result.returncode != 0:
            return f"[FAIL] {ts} {set_name} {HAR} {burden_class} | rc={result.returncode} | {result.stderr[-200:]}"
        return f"[DONE] {ts} {set_name} {HAR} {burden_class}"
    except Exception as e:
        return f"[ERR ] {ts} {set_name} {HAR} {burden_class} | {e}"


def _run_task(args):
    return burden_rvtest(*args)

## 6. Correr todos los burden tests para EUR (paralelo)

In [ ]:
# Construir lista de tareas: (set, HAR, class, freqUpper)
tasks = [
    (ds, HAR, cls_name, cls_freq)
    for ds in datasets
    for HAR in HARS_DICT
    for cls_name, cls_freq in BURDEN_CLASSES.items()
]
print(f"Total tareas: {len(tasks)}  ({len(datasets)} datasets x {len(HARS_DICT)} HARs x {len(BURDEN_CLASSES)} clases)")
print(f"Workers: {MAX_WORKERS}\n")

ts_start = datetime.now()
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_run_task, t): t for t in tasks}
    completed = 0
    for fut in as_completed(futures):
        completed += 1
        try:
            msg = fut.result()
        except Exception as e:
            msg = f"[EXC ] {futures[fut]} | {e}"
        # imprimir 1 de cada N para no spammear
        if completed % 50 == 0 or msg.startswith(('[FAIL', '[ERR ', '[EXC ')):
            print(f"[{completed}/{len(tasks)}] {msg}")

print(f"\nTerminado en {datetime.now() - ts_start}")

In [ ]:
def vcf_path(set_name, HAR):
    return f"{DIR_WSPS}/Working_{set_name}/{ANCESTRY}/InputFiles/Indiv_HARS/{HAR}.vcf.gz"

def covar_path(set_name):
    return f"{DIR_WSPS}/Working_{set_name}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"

summary = {}
for ds in datasets:
    cov = covar_path(ds)
    cov_ok = os.path.isfile(cov)
    missing_vcfs = []
    for HAR in HARS_DICT:
        v = vcf_path(ds, HAR)
        if not (os.path.isfile(v) and os.path.isfile(v + '.tbi')):
            missing_vcfs.append(HAR)
    summary[ds] = {
        'covariate_ok': cov_ok,
        'covariate_path': cov,
        'n_vcfs_present': len(HARS_DICT) - len(missing_vcfs),
        'n_vcfs_missing': len(missing_vcfs),
        'missing_examples': missing_vcfs[:5],
    }

for ds, s in summary.items():
    print(f"\n=== {ds} ===")
    for k, v in s.items():
        print(f"  {k}: {v}")

## 7. Compilar resultados (SKAT y SKAT-O)

Junta todos los `.assoc` por dataset/clase en dos TSVs (uno SKAT, uno SKAT-O), agregando columnas ANCESTRY, GENE (HAR), CLASS.

In [ ]:
def collect_assoc(set_name):
    skat_rows  = []
    skato_rows = []
    burden_dir = Path(f"{DIR_BURDEN}/{set_name}/{ANCESTRY}")

    for HAR in HARS_DICT:
        for cls in BURDEN_CLASSES:
            p_skat  = burden_dir / f"{HAR}_{cls}.burden.Skat.assoc"
            p_skato = burden_dir / f"{HAR}_{cls}.burden.SkatO.assoc"

            if p_skat.is_file():
                try:
                    df = pd.read_csv(p_skat, sep='\t')
                    df.insert(0, 'ANCESTRY', ANCESTRY)
                    df.insert(1, 'GENE', HAR)
                    df.insert(2, 'CLASS', cls)
                    df.insert(3, 'DATASET', set_name)
                    skat_rows.append(df)
                except Exception as e:
                    print(f"  warn read SKAT {HAR} {cls}: {e}")

            if p_skato.is_file():
                try:
                    df = pd.read_csv(p_skato, sep='\t')
                    df.insert(0, 'ANCESTRY', ANCESTRY)
                    df.insert(1, 'GENE', HAR)
                    df.insert(2, 'CLASS', cls)
                    df.insert(3, 'DATASET', set_name)
                    skato_rows.append(df)
                except Exception as e:
                    print(f"  warn read SKAT-O {HAR} {cls}: {e}")

    skat_df  = pd.concat(skat_rows,  ignore_index=True) if skat_rows  else pd.DataFrame()
    skato_df = pd.concat(skato_rows, ignore_index=True) if skato_rows else pd.DataFrame()
    return skat_df, skato_df


for ds in datasets:
    print(f"\n=== Compilando {ds} ===")
    skat_df, skato_df = collect_assoc(ds)
    out_skat  = f"{DIR_RESU}/{ds}_{ANCESTRY}_BURDEN.SKAT.tsv"
    out_skato = f"{DIR_RESU}/{ds}_{ANCESTRY}_BURDEN.SKAT-O.tsv"
    skat_df.to_csv(out_skat,   sep='\t', index=False)
    skato_df.to_csv(out_skato, sep='\t', index=False)
    print(f"  SKAT   : {len(skat_df):>5} filas → {out_skat}")
    print(f"  SKAT-O : {len(skato_df):>5} filas → {out_skato}")

## 8. Quick look: top hits

In [ ]:
for ds in datasets:
    for kernel in ['SKAT', 'SKAT-O']:
        path = f"{DIR_RESU}/{ds}_{ANCESTRY}_BURDEN.{kernel}.tsv"
        if not os.path.isfile(path):
            continue
        df = pd.read_csv(path, sep='\t')
        if df.empty or 'Pvalue' not in df.columns:
            continue
        print(f"\n--- Top 10 {ds} {kernel} ---")
        print(df.sort_values('Pvalue').head(10).to_string(index=False))